# Baselines: Bike Sharing (short)

Step 5 of `01_bike_sharing_project.ipynb`, continuing `eda_short.ipynb`. Two baselines, learned on the training set and scored with 5-fold cross-validation:

1. **Mean baseline:** `DummyRegressor` predicts the training mean for every hour.
2. **Rule-based baseline:** mean rentals per value of the feature with the strongest (Spearman) correlation to `rentals`.

Metrics: **RMSE** (main metric, same unit as the target, penalises large rush-hour errors), MAE, R².

In [1]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.utils.validation import check_is_fitted

pd.set_option("display.width", 200)

## 1. Data preparation (from `eda_short.ipynb`)

In [2]:
df = pd.read_csv("data/bike_sharing.csv", parse_dates=["datetime"])
df_train, df_test = train_test_split(df, test_size=0.2, random_state=42)  # df_test stays locked until step 8


def nth_sunday(year, month, n):
    """Date of the n-th Sunday of a month."""
    days = pd.date_range(f"{year}-{month:02d}-01", periods=31, freq="D")
    sundays = days[(days.month == month) & (days.dayofweek == 6)]
    return sundays[n - 1]


def dst_artifact_hours(data):
    """Boolean mask of the hours that are corrupt because of the clock change."""
    years = data["datetime"].dt.year.unique()
    spring_forward = [nth_sunday(y, 3, 2) + pd.Timedelta(hours=2) for y in years]  # 02:00 does not exist
    fall_back = [nth_sunday(y, 11, 1) + pd.Timedelta(hours=1) for y in years]      # 01:00 counts two hours
    return data["datetime"].isin(spring_forward + fall_back)


def remove_dst_artifacts(data):
    """Drop the non-existent 02:00 on spring-forward Sundays and the doubled 01:00 on fall-back Sundays."""
    return data[~dst_artifact_hours(data)].copy()


WEATHER_GROUPS = {
    0: "clear", 1: "clear", 2: "clear",
    3: "overcast",
    51: "drizzle", 53: "drizzle", 55: "drizzle",
    61: "rain", 63: "rain", 65: "rain",
    71: "snow", 73: "snow", 75: "snow",
}


def add_features(data):
    """Derive calendar features from `datetime` and `holiday`, and group the weather codes."""
    data = data.copy()
    data["hour"] = data["datetime"].dt.hour
    data["weekday"] = data["datetime"].dt.dayofweek  # 0 = Monday, 6 = Sunday
    data["month"] = data["datetime"].dt.month
    data["is_holiday"] = data["holiday"].notna().astype(int)
    data["day_type"] = np.select(
        [data["is_holiday"] == 1, data["weekday"] >= 5],
        ["holiday", "weekend"],
        default="working day",
    )
    data["weather_group"] = data["weather_code"].map(WEATHER_GROUPS)
    return data


LEAKAGE_COLUMNS = ["casual", "member"]


def prepare_data(data):
    """All manual changes to the raw data. Apply it unchanged to df_test in step 8."""
    data = remove_dst_artifacts(data)
    data = add_features(data)
    return data.drop(columns=LEAKAGE_COLUMNS + ["holiday"])


TARGET = "rentals"
df_train_prepared = prepare_data(df_train)
X_train = df_train_prepared.drop(columns=[TARGET, "datetime"])
y_train = df_train_prepared[TARGET]
print("X_train:", X_train.shape, "| y_train:", y_train.shape)

X_train: (14013, 14) | y_train: (14013,)


## 2. Cross-validation

In [3]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)
SCORING = {"RMSE": "neg_root_mean_squared_error", "MAE": "neg_mean_absolute_error", "R2": "r2"}


def cv_scores(model, X, y, name):
    """Cross-validated training and validation scores (mean over the folds) as one table row."""
    res = cross_validate(model, X, y, cv=cv, scoring=SCORING, return_train_score=True)
    row = {}
    for metric in SCORING:
        sign = 1 if metric == "R2" else -1  # sklearn returns negated errors
        row[f"train_{metric}"] = sign * res[f"train_{metric}"].mean()
        row[f"cv_{metric}"] = sign * res[f"test_{metric}"].mean()
    return pd.Series(row, name=name)

## 3. Baseline 1: mean

In [4]:
baseline_mean = DummyRegressor(strategy="mean").fit(X_train, y_train)
print(f"Constant prediction: {baseline_mean.constant_[0][0]:.1f} rentals per hour")

Constant prediction: 753.4 rentals per hour


## 4. Baseline 2: mean per value of the most strongly correlated feature

In [5]:
numeric_features = X_train.select_dtypes("number").columns
spearman = X_train[numeric_features].corrwith(y_train, method="spearman")
spearman = spearman.reindex(spearman.abs().sort_values(ascending=False).index)

best_feature = spearman.index[0]
print(f"Strongest correlation: {best_feature!r} (Spearman rho = {spearman.iloc[0]:.3f})")
spearman.head(4).round(3).to_frame("spearman").T

Strongest correlation: 'hour' (Spearman rho = 0.547)


,hour,temperature,feels_like,humidity
spearman,0.547,0.474,0.428,-0.411


In [6]:
class GroupMeanRegressor(RegressorMixin, BaseEstimator):
    """Rule-based baseline: predict the mean target of the training rows that share the same value of `feature`."""

    def __init__(self, feature="hour"):
        self.feature = feature

    def fit(self, X, y):
        y = pd.Series(np.asarray(y), index=X.index)
        self.means_ = y.groupby(X[self.feature]).mean()
        self.global_mean_ = y.mean()  # fallback for values not seen in training
        return self

    def predict(self, X):
        check_is_fitted(self, "means_")
        return X[self.feature].map(self.means_).fillna(self.global_mean_).to_numpy()


baseline_rule = GroupMeanRegressor(feature=best_feature).fit(X_train, y_train)
rule_values = baseline_rule.means_.round(0).astype(int).rename("predicted rentals")
rule_values.to_frame().T

hour,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
predicted rentals,218,137,80,44,43,121,317,685,1157,862,...,1065,1300,1463,1783,1557,1187,878,703,556,376


## 5. Results

In [7]:
baseline_scores = pd.DataFrame([
    cv_scores(DummyRegressor(strategy="mean"), X_train, y_train, "Mean baseline"),
    cv_scores(GroupMeanRegressor(feature=best_feature), X_train, y_train, f"Rule baseline (mean per {best_feature})"),
])
baseline_scores.round(3)

,train_RMSE,cv_RMSE,train_MAE,cv_MAE,train_R2,cv_R2
Mean baseline,636.223,636.227,521.955,522.059,0.000,-0.001
Rule baseline (mean per hour),393.568,394.934,278.336,279.195,0.617,0.614


- **Mean baseline:** CV RMSE ≈ 636 (= standard deviation of `rentals`), R² ≈ 0.
- **Rule baseline (mean per `hour`):** CV RMSE ≈ 395 (−38 %), R² ≈ 0.61. Misses the different daily patterns of working days vs. weekends/holidays, and knows nothing about weather or season.
- Training ≈ CV scores for both: no overfitting. Every model in steps 6 and 7 has to beat the **rule baseline**.
- Kept for step 8: `baseline_mean`, `baseline_rule`, `rule_values`, `baseline_scores`.